In [2]:
import os
import zipfile
import numpy as np
import tensorflow as tf

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalAveragePooling1D,
    GlobalMaxPooling1D,
    BatchNormalization,
    ReLU,
    Add,
    Dense,
    Multiply,
    Concatenate,
    Reshape,
    Activation,
    Lambda
)

from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 1. Seed
# ============================================================

SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. UCI-HAR Path
# ============================================================

DATA_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "UCI HAR Dataset"
)

# ============================================================
# 3. Download
# ============================================================

if not os.path.exists(DATA_PATH):

    print("Downloading UCI-HAR dataset...")

    url = (
        "https://archive.ics.uci.edu/ml/"
        "machine-learning-databases/00240/"
        "UCI%20HAR%20Dataset.zip"
    )

    zip_path = tf.keras.utils.get_file(
        "UCI_HAR.zip",
        origin=url,
        cache_dir="/content",
        cache_subdir=""
    )

    extract_path = (
        "/content/drive/MyDrive/Colab Notebooks"
    )

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_path)

else:
    print("Existing dataset found")


# ============================================================
# 4. Signal Names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]


# ============================================================
# 5. Load Dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/"
            f"{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)
        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # Class labels: 1~6 -> 0~5
    y = y - 1

    return X.astype(np.float32), y


X_train, y_train = load_data("train")
X_test, y_test = load_data("test")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


# ============================================================
# 6. Normalization
# ============================================================

mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (X_train - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)


# ============================================================
# 7. One-hot Encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_test_onehot = to_categorical(
    y_test,
    num_classes=n_outputs
)


# ============================================================
# 8. CBAM: Channel Attention
# ============================================================

def channel_attention(x, reduction_ratio=8):

    channels = int(x.shape[-1])

    hidden_units = max(
        channels // reduction_ratio,
        1
    )

    # Average pooling across time
    avg_pool = GlobalAveragePooling1D()(x)

    # Max pooling across time
    max_pool = GlobalMaxPooling1D()(x)

    # Shared MLP
    shared_dense_1 = Dense(
        hidden_units,
        activation="relu",
        use_bias=True
    )

    shared_dense_2 = Dense(
        channels,
        use_bias=True
    )

    avg_out = shared_dense_2(
        shared_dense_1(avg_pool)
    )

    max_out = shared_dense_2(
        shared_dense_1(max_pool)
    )

    # Combine average and max descriptors
    attention = Add()([
        avg_out,
        max_out
    ])

    attention = Activation("sigmoid")(attention)

    # (batch, channels) -> (batch, 1, channels)
    attention = Reshape((1, channels))(attention)

    # Channel-wise feature refinement
    out = Multiply()([x, attention])

    return out


# ============================================================
# 9. CBAM: Spatial / Temporal Attention
# ============================================================

def spatial_attention(x, kernel_size=7):

    # Average across channels at each time step
    avg_pool = Lambda(
        lambda t: tf.reduce_mean(
            t, axis=-1, keepdims=True
        )
    )(x)

    # Max across channels at each time step
    max_pool = Lambda(
        lambda t: tf.reduce_max(
            t, axis=-1, keepdims=True
        )
    )(x)

    # Shape: (batch, time_steps, 2)
    concat = Concatenate(axis=-1)([
        avg_pool,
        max_pool
    ])

    # Learn attention weights along the time axis
    attention = Conv1D(
        filters=1,
        kernel_size=kernel_size,
        padding="same",
        activation="sigmoid",
        use_bias=False
    )(concat)

    # Temporal feature refinement
    out = Multiply()([x, attention])

    return out


# ============================================================
# 10. Complete CBAM Module
# ============================================================

def cbam_block(
    x,
    reduction_ratio=8,
    spatial_kernel=7
):

    x = channel_attention(
        x,
        reduction_ratio=reduction_ratio
    )

    x = spatial_attention(
        x,
        kernel_size=spatial_kernel
    )

    return x


# ============================================================
# 11. CBAM Residual Block
# ============================================================

def cbam_residual_block(
    x,
    filters,
    kernel_size=3,
    reduction_ratio=8,
    spatial_kernel=7
):

    shortcut = x

    # Main path
    x = Conv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)
    x = ReLU()(x)

    x = Conv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False
    )(x)

    x = BatchNormalization()(x)

    # Adjust shortcut dimensions if necessary
    if int(shortcut.shape[-1]) != filters:

        shortcut = Conv1D(
            filters,
            kernel_size=1,
            padding="same",
            use_bias=False
        )(shortcut)

        shortcut = BatchNormalization()(shortcut)

    # Residual connection
    x = Add()([
        x,
        shortcut
    ])

    x = ReLU()(x)

    # Apply CBAM attention
    x = cbam_block(
        x,
        reduction_ratio=reduction_ratio,
        spatial_kernel=spatial_kernel
    )

    return x


# ============================================================
# 12. Build CBAM Model
# ============================================================

inputs = Input(
    shape=(128, 9),
    name="input_signal"
)

# Initial convolution
x = Conv1D(
    32,
    kernel_size=3,
    padding="same",
    use_bias=False
)(inputs)

x = BatchNormalization()(x)
x = ReLU()(x)


# Residual Block 1: 32 filters
x = cbam_residual_block(
    x,
    filters=32
)

x = MaxPooling1D(
    pool_size=2
)(x)


# Residual Block 2: 64 filters
x = cbam_residual_block(
    x,
    filters=64
)

x = MaxPooling1D(
    pool_size=2
)(x)


# Residual Block 3: 128 filters
x = cbam_residual_block(
    x,
    filters=128
)


# Classification head
x = GlobalAveragePooling1D()(x)

x = Dense(
    128,
    activation="relu"
)(x)

outputs = Dense(
    n_outputs,
    activation="softmax",
    name="classification"
)(x)


model = Model(
    inputs=inputs,
    outputs=outputs,
    name="CBAM_1D_HAR"
)


# ============================================================
# 13. Compile
# ============================================================

model.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    metrics=["accuracy"]
)

model.summary()


# ============================================================
# 14. Train
# ============================================================

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


# ============================================================
# 15. Prediction
# ============================================================

y_prob = model.predict(
    X_test,
    verbose=0
)

y_pred = np.argmax(
    y_prob,
    axis=1
)


# ============================================================
# 16. Evaluation
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)


print("\n==============================")
print("CBAM TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")


# ============================================================
# 17. Classification Report
# ============================================================

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 18. Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix")
print(cm)

Existing dataset found
X_train: (7352, 128, 9)
X_test : (2947, 128, 9)


Model: "CBAM_1D_HAR"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_signal        │ (None, 128, 9)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_12 (Conv1D)  │ (None, 128, 32)   │        864 │ input_signal[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_12[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_7 (ReLU)      │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 128, 32)   │      3,072 │ re_lu_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_13[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_8 (ReLU)      │ (None, 128, 32)   │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 128, 32)   │      3,072 │ re_lu_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 32)   │        128 │ conv1d_14[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 128, 32)   │          0 │ batch_normalizat… │
│                     │                   │            │ re_lu_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_9 (ReLU)      │ (None, 128, 32)   │          0 │ add_6[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ re_lu_9[0][0]     │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 32)        │          0 │ re_lu_9[0][0]     │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 4)         │        132 │ global_average_p… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 32)        │        160 │ dense_7[0][0],    │
│                     │                   │            │ dense_7[1][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_7 (Add)         │ (None, 32)        │          0 │ dense_8[0][0],    │
│                     │                   │            │ dense_8[1][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 32)        │          0 │ add_7[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 1, 32)     │          0 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_6          │ (None, 128, 32)   │          0 │ re_lu_9[0][0],    │
│ (Multiply)          │                   │            │ reshape_3[0][0] 

 Total params: 135,052 (527.55 KB)

 Trainable params: 133,708 (522.30 KB)

 Non-trainable params: 1,344 (5.25 KB)

Epoch 1/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 24s 126ms/step - accuracy: 0.8973 - loss: 0.3098 - val_accuracy: 0.2481 - val_loss: 1.9078
Epoch 2/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.9539 - loss: 0.1125 - val_accuracy: 0.2733 - val_loss: 2.1663
Epoch 3/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9589 - loss: 0.0971 - val_accuracy: 0.4731 - val_loss: 1.9488
Epoch 4/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9645 - loss: 0.0846 - val_accuracy: 0.5846 - val_loss: 1.1535
Epoch 5/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9665 - loss: 0.0799 - val_accuracy: 0.8477 - val_loss: 0.4449
Epoch 6/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9645 - loss: 0.0777 - val_accuracy: 0.9028 - val_loss: 0.3289
Epoch 7/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9701 - loss: 0.0694 - val_accuracy: 0.9205 - val_loss: 0.3298
Epoch 8/20
92/92 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9735 - loss: 0.0633 - val_accuracy: 0.9198 - val_